# Notebook 03 — Edit Distance and the Alignment Problem

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 03 of 18  
**Time estimate:** 75 minutes

> Edit distance is the intellectual scaffolding for NW alignment. Master this
> first, then NB04's additional complexity is one small step.

---
## Step 1 — Motivation

How similar are two DNA sequences? This question appears constantly: is this read from
the human genome or a contaminant? Are these two proteins homologous? Edit distance
is the simplest mathematical formulation of sequence similarity, and its dynamic
programming solution is the direct ancestor of Needleman-Wunsch.

---
## Step 2 — Intuition

How many single-character changes (insertions, deletions, substitutions) does it take
to transform one string into another? That is the Levenshtein edit distance.

"KITTEN" → "SITTING" requires 3 changes:
1. K→S (substitution)
2. E→I (substitution)
3. insert G at end (insertion)

For biological sequences, the three operations map directly to:
- **Substitution** → point mutation
- **Insertion** → insertion event (gaps in one sequence)
- **Deletion** → deletion event (gaps in the other sequence)

The key insight: *optimal* edit distance can be computed by dynamic programming.
Trying all possible edit paths is exponential; DP is O(n×m).

---
## Step 3 — Biological Background

Mutations accumulate over evolutionary time. Two sequences that diverged from a
common ancestor will have accumulated substitutions, insertions, and deletions. The
edit distance (or its generalization, alignment score) quantifies this divergence.

Important: not all edits are equally likely. Transitions (A↔G, C↔T) are more
frequent than transversions. Indels are often in runs. Simple edit distance treats
all three operations as cost 1 — which is why NW uses a scoring matrix instead.

---
## Step 4 — Mathematical Explanation

**Levenshtein recurrence:**

Let $d[i][j]$ = edit distance between $s_1[1..i]$ and $s_2[1..j]$.

**Base cases:**
$$d[0][j] = j \quad (\text{delete j characters from } s_2)$$
$$d[i][0] = i \quad (\text{insert i characters into empty string})$$

**Recurrence:**
$$d[i][j] = \begin{cases}
d[i-1][j-1] & \text{if } s_1[i] = s_2[j] \quad (\text{no edit needed}) \\
1 + \min\begin{cases}
  d[i-1][j] & (\text{delete from } s_1) \\
  d[i][j-1] & (\text{insert into } s_1) \\
  d[i-1][j-1] & (\text{substitute})
\end{cases} & \text{otherwise}
\end{cases}$$

**Complexity:** O(n×m) time, O(n×m) space (reducible to O(min(n,m)) space if only the distance, not the traceback, is needed).

---
## Step 5 — Computational Explanation

```
Initialize d[0][j] = j, d[i][0] = i

For i in 1..len(s1):
    For j in 1..len(s2):
        if s1[i-1] == s2[j-1]:           # 0-indexed sequences
            d[i][j] = d[i-1][j-1]
        else:
            d[i][j] = 1 + min(
                d[i-1][j],    # delete
                d[i][j-1],    # insert
                d[i-1][j-1]  # substitute
            )

Return d[len(s1)][len(s2)]
```

Traceback: from `d[n][m]`, walk backward: if came from diagonal match → match/mismatch; if from left → insert; if from above → delete.

In [ ]:
# Step 6 — Python Implementation
import numpy as np
from typing import Tuple


def edit_distance(s1: str, s2: str) -> int:
    n, m = len(s1), len(s2)
    d = np.zeros((n + 1, m + 1), dtype=int)

    # Base cases
    for i in range(n + 1):
        d[i, 0] = i
    for j in range(m + 1):
        d[0, j] = j

    # Fill
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                d[i, j] = d[i - 1, j - 1]
            else:
                d[i, j] = 1 + min(
                    d[i - 1, j],    # delete from s1
                    d[i, j - 1],    # insert into s1
                    d[i - 1, j - 1] # substitute
                )
    return int(d[n, m])


def edit_distance_with_traceback(s1: str, s2: str) -> Tuple[int, str, str]:
    n, m = len(s1), len(s2)
    d = np.zeros((n + 1, m + 1), dtype=int)
    backtrack = np.full((n + 1, m + 1), '', dtype=object)

    for i in range(n + 1):
        d[i, 0] = i
        backtrack[i, 0] = 'U'  # Up (delete from s1)
    for j in range(m + 1):
        d[0, j] = j
        backtrack[0, j] = 'L'  # Left (insert into s1)
    backtrack[0, 0] = 'S'  # Start

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                d[i, j] = d[i - 1, j - 1]
                backtrack[i, j] = 'D'  # Diagonal match
            else:
                options = {
                    'U': d[i - 1, j],    # delete
                    'L': d[i, j - 1],    # insert
                    'D': d[i - 1, j - 1] # substitute
                }
                best = min(options, key=options.get)
                d[i, j] = 1 + options[best]
                backtrack[i, j] = best

    # Traceback to build alignment
    align1, align2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        move = backtrack[i, j]
        if move == 'D':
            align1.append(s1[i - 1])
            align2.append(s2[j - 1])
            i -= 1; j -= 1
        elif move == 'U':
            align1.append(s1[i - 1])
            align2.append('-')
            i -= 1
        elif move == 'L':
            align1.append('-')
            align2.append(s2[j - 1])
            j -= 1
        else:
            break

    return int(d[n, m]), ''.join(reversed(align1)), ''.join(reversed(align2))


# Tests
print("Tests:")
print(edit_distance('KITTEN', 'SITTING'), '(expect 3)')
print(edit_distance('ATCG', 'ATCG'), '(expect 0)')
print(edit_distance('ATCG', 'TTCG'), '(expect 1)')
print(edit_distance('ATCG', 'ATCGTT'), '(expect 2)')
print(edit_distance('', 'ATCG'), '(expect 4)')

print()
dist, a1, a2 = edit_distance_with_traceback('ATCGATCG', 'ATGGATCG')
print(f"Edit distance: {dist}")
print(f"Seq1: {a1}")
print(f"Seq2: {a2}")

In [ ]:
# Step 7 — Visualization: the DP matrix
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def visualize_edit_distance_matrix(s1: str, s2: str):
    n, m = len(s1), len(s2)
    d = np.zeros((n + 1, m + 1), dtype=int)
    backtrack = np.full((n + 1, m + 1), '', dtype=object)

    for i in range(n + 1):
        d[i, 0] = i
        backtrack[i, 0] = 'U'
    for j in range(m + 1):
        d[0, j] = j
        backtrack[0, j] = 'L'
    backtrack[0, 0] = 'S'

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s1[i - 1] == s2[j - 1]:
                d[i, j] = d[i - 1, j - 1]
                backtrack[i, j] = 'D'
            else:
                options = {'U': d[i-1,j], 'L': d[i,j-1], 'D': d[i-1,j-1]}
                best = min(options, key=options.get)
                d[i, j] = 1 + options[best]
                backtrack[i, j] = best

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: the DP matrix
    ax = axes[0]
    im = ax.imshow(d, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(m + 1))
    ax.set_xticklabels([' '] + list(s2), fontsize=11)
    ax.set_yticks(range(n + 1))
    ax.set_yticklabels([' '] + list(s1), fontsize=11)
    for i in range(n + 1):
        for j in range(m + 1):
            ax.text(j, i, d[i, j], ha='center', va='center', fontsize=9,
                    color='black' if d[i, j] < d.max() * 0.7 else 'white')
    ax.set_title(f'Edit distance DP matrix\n"{s1}" vs "{s2}" → d={d[n,m]}')
    plt.colorbar(im, ax=ax)

    # Right: traceback path
    ax2 = axes[1]
    ax2.imshow(d, cmap='Blues', alpha=0.3, aspect='auto')
    ax2.set_xticks(range(m + 1))
    ax2.set_xticklabels([' '] + list(s2), fontsize=11)
    ax2.set_yticks(range(n + 1))
    ax2.set_yticklabels([' '] + list(s1), fontsize=11)

    # Trace path
    i, j = n, m
    path_cells = [(i, j)]
    while i > 0 or j > 0:
        move = backtrack[i, j]
        if move == 'D': i -= 1; j -= 1
        elif move == 'U': i -= 1
        elif move == 'L': j -= 1
        else: break
        path_cells.append((i, j))

    px = [c[1] for c in path_cells]
    py = [c[0] for c in path_cells]
    ax2.plot(px, py, 'ro-', markersize=10, linewidth=2, label='Optimal path')
    ax2.plot(px[0], py[0], 'g*', markersize=15, label='End')
    ax2.plot(px[-1], py[-1], 'bs', markersize=12, label='Start')
    for i_, j_ in path_cells:
        ax2.text(j_, i_, d[i_, j_], ha='center', va='center', fontsize=8)
    ax2.set_title(f'Traceback path (edit distance = {d[n,m]})')
    ax2.legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig('edit_distance_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


visualize_edit_distance_matrix('ATCGAT', 'ATCGGT')

In [ ]:
# Biological application: compare edit distance of diverging sequences
import matplotlib.pyplot as plt

rng = np.random.default_rng(99)

def mutate_sequence(seq: str, mutation_rate: float, rng) -> str:
    bases = list('ACGT')
    result = list(seq)
    for i in range(len(result)):
        if rng.random() < mutation_rate:
            result[i] = rng.choice([b for b in bases if b != result[i]])
    return ''.join(result)

# Start with an ancestral sequence
ancestor = ''.join(rng.choice(list('ACGT'), 100))
rates = np.linspace(0, 0.5, 20)
distances = []

for rate in rates:
    evolved = mutate_sequence(ancestor, rate, rng)
    d = edit_distance(ancestor, evolved)
    distances.append(d)

plt.figure(figsize=(7, 4))
plt.plot(rates, distances, 'bo-', markersize=5)
plt.plot(rates, [r * len(ancestor) for r in rates], 'r--', label='Expected (no saturation)')
plt.xlabel('Mutation rate per site')
plt.ylabel('Edit distance')
plt.title('Edit distance vs. mutation rate (100 bp sequence)\nSaturation visible at high rates')
plt.legend()
plt.tight_layout()
plt.savefig('edit_distance_mutation_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("Note: at high mutation rates, edit distance saturates (multiple hits).")
print("This is why JC/K2P distance corrections exist (see Module 06).")

---
## Step 8 — Exercises

See `exercises/03_edit_distance_exercises.md`.

1. Implement a **space-efficient** edit distance that uses only two rows (current and
   previous) instead of the full matrix. Verify it gives the same distance.
2. Implement **Hamming distance** (only substitutions, no indels; only valid for equal-
   length strings). How does it differ from Levenshtein?
3. Write a function to count the number of **optimal edit paths** (not just one).
4. What is the edit distance between 'ACGT' and 'TGCA'? Walk through the DP matrix
   by hand (draw the table).

---
## Step 10 — Quiz

1. What three operations define Levenshtein edit distance?
2. What is the time complexity of the DP algorithm?
3. Can edit distance be 0 for two different strings? Explain.
4. Why does edit distance saturate at high mutation rates?
5. How does edit distance differ from the alignment score used in NW/SW?

---
## Step 12 — Reflection

> *[Explain in one paragraph why dynamic programming is better than trying all possible
> edit paths for two sequences of length 100. What is the exponential approach, and why
> does DP avoid it?]*

---
## Step 13 — Papers Referenced

- Levenshtein (1966) "Binary codes capable of correcting deletions, insertions, and
  reversals." *Soviet Physics Doklady* — the original edit distance paper.

---
*Next: `04_needleman_wunsch_global_alignment.ipynb`*